# Proyecto Integrador — Semana 02  
# 02 Silver — Daniel Guzmán

## Objetivo

Leer exclusivamente desde las tablas Bronze, limpiar tipos, estandarizar nombres, construir una tabla maestra enriquecida y documentar el impacto de cada JOIN.

## Entradas Bronze

- `workspace.bronze.transactions_daniel`
- `workspace.bronze.users_daniel`
- `workspace.bronze.cards_daniel`
- `workspace.bronze.mcc_codes_daniel`
- `workspace.bronze.fraud_labels_daniel`

## Salida Silver

- `workspace.silver.transactions_daniel`

## Transformaciones principales

- Normalización de nombres de columnas.
- Conversión de `amount` a número.
- Creación de `amount_abs`.
- Conversión de `date` a `transaction_date`.
- Creación de columnas temporales: `hora`, `dia_semana`, `es_fin_de_semana`, `mes`, `anio`.
- Conversión de MCC de formato ancho a formato largo.
- Conversión de fraude `Yes/No` a `1/0`.
- JOIN de las 5 tablas.
- Validación de pérdidas y duplicados.


In [0]:
from pyspark.sql import functions as F
import re

MI_NOMBRE = "daniel"
CATALOG = "workspace"

spark.sql(f"USE CATALOG {CATALOG}")
spark.sql("CREATE SCHEMA IF NOT EXISTS silver")

df_tx = spark.table(f"{CATALOG}.bronze.transactions_{MI_NOMBRE}")
df_users = spark.table(f"{CATALOG}.bronze.users_{MI_NOMBRE}")
df_cards = spark.table(f"{CATALOG}.bronze.cards_{MI_NOMBRE}")
df_mcc_raw = spark.table(f"{CATALOG}.bronze.mcc_codes_{MI_NOMBRE}")
df_fraud = spark.table(f"{CATALOG}.bronze.fraud_labels_{MI_NOMBRE}")

print("Tablas Bronze cargadas:")
for nombre, dataframe in [
    ("transactions", df_tx),
    ("users", df_users),
    ("cards", df_cards),
    ("mcc_codes", df_mcc_raw),
    ("fraud_labels", df_fraud)
]:
    print(f"{nombre}: {dataframe.count():,} filas | {len(dataframe.columns)} columnas")

In [0]:
def to_snake_case(col_name):
    col_name = col_name.strip().lower()
    col_name = re.sub(r"[^a-z0-9]+", "_", col_name)
    col_name = re.sub(r"_+", "_", col_name).strip("_")
    return col_name

def normalize_columns(df):
    for old_col in df.columns:
        df = df.withColumnRenamed(old_col, to_snake_case(old_col))
    return df

df_tx = normalize_columns(df_tx)
df_users = normalize_columns(df_users)
df_cards = normalize_columns(df_cards)
df_fraud = normalize_columns(df_fraud)

print("Columnas normalizadas correctamente")
print("transactions:", df_tx.columns)
print("users:", df_users.columns)
print("cards:", df_cards.columns)
print("fraud:", df_fraud.columns)

In [0]:
df_tx_clean = (
    df_tx
    .withColumnRenamed("id", "transaction_id")
    .withColumnRenamed("client_id", "user_id")
    .withColumnRenamed("ingested_at", "tx_ingested_at")
    .withColumn("amount", F.regexp_replace(F.col("amount"), "[$,]", "").cast("double"))
    .withColumn("amount_abs", F.abs(F.col("amount")))
    .withColumn("transaction_date", F.to_timestamp(F.col("date"), "yyyy-MM-dd HH:mm:ss"))
    .withColumn("hora", F.hour("transaction_date"))
    .withColumn("dia_semana", F.dayofweek("transaction_date"))
    .withColumn("es_fin_de_semana", F.when(F.col("dia_semana").isin([1, 7]), 1).otherwise(0))
    .withColumn("mes", F.month("transaction_date"))
    .withColumn("anio", F.year("transaction_date"))
    .withColumn("mcc", F.col("mcc").cast("int"))
    .withColumn("user_id", F.col("user_id").cast("int"))
    .withColumn("card_id", F.col("card_id").cast("int"))
    .drop("date")
)

display(df_tx_clean.limit(5))
print(f"Transactions limpias: {df_tx_clean.count():,}")

In [0]:
df_users_clean = (
    df_users
    .drop("ingested_at")
    .withColumnRenamed("id", "user_id")
    .withColumn("user_id", F.col("user_id").cast("int"))
)

df_cards_clean = (
    df_cards
    .drop("ingested_at")
    .withColumnRenamed("id", "card_id")
    .withColumnRenamed("client_id", "card_user_id")
    .withColumn("card_id", F.col("card_id").cast("int"))
    .withColumn("card_user_id", F.col("card_user_id").cast("int"))
    .withColumn("credit_limit_num", F.regexp_replace(F.col("credit_limit"), "[$,]", "").cast("double"))
)

print("Users clean:")
df_users_clean.printSchema()

print("Cards clean:")
df_cards_clean.printSchema()

In [0]:
mcc_cols = [c for c in df_mcc_raw.columns if c != "_ingested_at"]

stack_expr = (
    f"stack({len(mcc_cols)}, "
    + ", ".join([f"'{c}', `{c}`" for c in mcc_cols])
    + ") as (mcc_str, merchant_category)"
)

df_mcc_clean = (
    df_mcc_raw
    .select(F.expr(stack_expr))
    .withColumn("mcc", F.col("mcc_str").cast("int"))
    .drop("mcc_str")
)

print(f"Categorías MCC: {df_mcc_clean.count():,}")
display(df_mcc_clean.limit(10))

In [0]:
df_fraud_clean = (
    df_fraud
    .withColumnRenamed("id", "transaction_id")
    .withColumnRenamed("target", "is_fraud_label")
    .withColumn(
        "is_fraud",
        F.when(F.col("is_fraud_label") == "Yes", 1)
         .when(F.col("is_fraud_label") == "No", 0)
         .otherwise(None)
    )
    .select("transaction_id", "is_fraud_label", "is_fraud")
)

display(df_fraud_clean.limit(5))

print("Distribución de fraude:")
df_fraud_clean.groupBy("is_fraud_label", "is_fraud").count().show()

In [0]:
tx_base = df_tx_clean.count()

df_join_users = df_tx_clean.join(df_users_clean, on="user_id", how="left")
perdidos_users = tx_base - df_join_users.count()
sin_users = df_join_users.filter(F.col("gender").isNull()).count()

df_join_cards = df_join_users.join(df_cards_clean, on="card_id", how="left")
perdidos_cards = df_join_users.count() - df_join_cards.count()
sin_cards = df_join_cards.filter(F.col("card_type").isNull()).count()

df_join_mcc = df_join_cards.join(df_mcc_clean, on="mcc", how="left")
perdidos_mcc = df_join_cards.count() - df_join_mcc.count()
sin_mcc = df_join_mcc.filter(F.col("merchant_category").isNull()).count()

df_silver = df_join_mcc.join(df_fraud_clean, on="transaction_id", how="left")
perdidos_fraud = df_join_mcc.count() - df_silver.count()
sin_fraud_label = df_silver.filter(F.col("is_fraud").isNull()).count()

print(f"Base transactions: {tx_base:,}")
print(f"Después JOIN users: {df_join_users.count():,} | perdidos: {perdidos_users:,} | sin datos users: {sin_users:,}")
print(f"Después JOIN cards: {df_join_cards.count():,} | perdidos: {perdidos_cards:,} | sin datos cards: {sin_cards:,}")
print(f"Después JOIN mcc: {df_join_mcc.count():,} | perdidos: {perdidos_mcc:,} | sin categoría MCC: {sin_mcc:,}")
print(f"Después JOIN fraud: {df_silver.count():,} | perdidos: {perdidos_fraud:,} | sin etiqueta fraude: {sin_fraud_label:,}")

display(df_silver.limit(5))

In [0]:
dup_count = (
    df_silver
    .groupBy("transaction_id")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print(f"Duplicados por transaction_id después de JOINs: {dup_count:,}")

df_silver.select(
    F.count("*").alias("total_registros"),
    F.sum(F.when(F.col("transaction_id").isNull(), 1).otherwise(0)).alias("transaction_id_null"),
    F.sum(F.when(F.col("user_id").isNull(), 1).otherwise(0)).alias("user_id_null"),
    F.sum(F.when(F.col("card_id").isNull(), 1).otherwise(0)).alias("card_id_null"),
    F.sum(F.when(F.col("amount").isNull(), 1).otherwise(0)).alias("amount_null"),
    F.sum(F.when(F.col("transaction_date").isNull(), 1).otherwise(0)).alias("transaction_date_null"),
    F.sum(F.when(F.col("merchant_category").isNull(), 1).otherwise(0)).alias("merchant_category_null"),
    F.sum(F.when(F.col("is_fraud").isNull(), 1).otherwise(0)).alias("is_fraud_null")
).show()

In [0]:
df_silver.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.silver.transactions_{MI_NOMBRE}")

print(f"Tabla Silver guardada: {CATALOG}.silver.transactions_{MI_NOMBRE}")
print(f"Filas: {df_silver.count():,}")
print(f"Columnas: {len(df_silver.columns)}")

## Documentación Silver

La tabla `workspace.silver.transactions_daniel` se construyó leyendo exclusivamente desde Bronze.

### Decisiones de limpieza

- Se normalizaron nombres de columnas para evitar problemas de espacios, mayúsculas y caracteres especiales.
- `amount` se convirtió a `double`.
- Se creó `amount_abs` para análisis de volumen sin eliminar montos negativos.
- `date` se convirtió a `transaction_date`.
- Se crearon columnas temporales: `hora`, `dia_semana`, `es_fin_de_semana`, `mes` y `anio`.
- MCC se transformó de formato ancho a formato largo.
- `target` se convirtió a `is_fraud_label` y a `is_fraud` numérico.

### Impacto de JOINs

Los JOINs se hicieron con `left join` para conservar todas las transacciones originales.  
De esta manera, si una dimensión no tiene coincidencia, la transacción no se pierde, sino que queda con valores nulos en esa dimensión.

Las transacciones sin etiqueta de fraude se mantienen porque no necesariamente son inválidas; simplemente no forman parte del set etiquetado. En Gold, las tasas de fraude deben calcularse usando solo transacciones etiquetadas.

### Columnas eliminadas

La columna original `date` se eliminó después de crear `transaction_date`, porque la nueva columna tiene tipo timestamp y permite análisis temporal correctamente.

### Ajuste técnico de columnas de ingesta

Como Bronze agrega `_ingested_at` a todas las tablas, al normalizar nombres y hacer JOINs varias tablas quedaban con la columna `ingested_at`.

Para evitar columnas duplicadas al guardar en Delta, se conservó la trazabilidad principal de `transactions` como `tx_ingested_at` y se eliminó `ingested_at` de las tablas dimensión antes del JOIN.